# Chapter 7. Satellite-Based Urban Analysis

*Starting With What You Have: A Quantitative Field Guide for Urban Research in Data-Scarce Settings*

Runs in a browser with no installation. Open in Google Colab and choose Runtime, then Run all.


## Step 0. Installation

To acquire real imagery rather than use the sample, see the Earth Engine scripts in `gee/`. Earth Engine runs in a browser and transfers only results, which matters where connectivity is constrained.

In [ ]:
!pip install -q rasterio scikit-learn matplotlib

## Step 1. Load the scenes

Landsat Collection 2 Level 2 is stored as integers and has to be converted back to physical quantities. Check immediately that reflectance is between 0 and 1 and temperatures are plausible.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import rs_toolkit as rs

s15, prof = rs.read_scene("data/scene_2015.tif")
s23, _ = rs.read_scene("data/scene_2023.tif")

print("bands:", list(s15.keys()))
print("reflectance range:", round(float(s15["red"].min()), 3), "to", round(float(s15["red"].max()), 3))
print("surface temperature range:", round(float(s23["lst_c"].min()), 1), "to",
      round(float(s23["lst_c"].max()), 1), "C")

## Step 2. Compute the spectral indices

Each index is a normalised difference of two bands, keeping values between -1 and +1. Confirm the indices agree with one another before trusting any of them.

In [ ]:
for label, fn in [("NDVI (vegetation)", rs.ndvi), ("NDBI (built-up)", rs.ndbi)]:
    a, b = float(np.nanmean(fn(s15))), float(np.nanmean(fn(s23)))
    print(f"{label:20s} 2015 {a:+.3f}   2023 {b:+.3f}   change {b - a:+.3f}")

t15, t23 = float(np.nanmean(s15["lst_c"])), float(np.nanmean(s23["lst_c"]))
print(f"{'surface temperature':20s} 2015 {t15:.2f}C  2023 {t23:.2f}C  change {t23 - t15:+.2f}C")

## Step 3. Supervised classification

Reporting overall accuracy and Kappa is obligatory. The very high accuracy here is an artefact of synthetic data; **on real imagery 0.85 to 0.92 is typical.**

In [ ]:
import rasterio

with rasterio.open("data/landcover_truth.tif") as src:
    truth = src.read()

lc15, metrics, clf = rs.classify(s15, truth[0])
print("overall accuracy:", round(metrics["overall_accuracy"], 3))
print("kappa:", round(metrics["kappa"], 3))

## Step 4. Change detection, using the same classifier

Apply the **same** classifier to both dates. Classify each separately and there is no way to tell real change from classification error.

In [ ]:
from sklearn.metrics import accuracy_score

lc23 = clf.predict(rs.build_features(s23)).reshape(truth[1].shape)
print("2023 prediction accuracy:", round(accuracy_score(truth[1].ravel(), lc23.ravel()), 3))

## Step 5. The transition matrix

A transition matrix says far more than net change. Simultaneous conversion of farmland into both formal and informal settlement calls for two different responses, and only the matrix shows both are happening.

In [ ]:
NAMES = ["water", "forest", "farmland", "formal", "informal", "bare"]
cm = rs.change_matrix(lc15, lc23, NAMES)
print(cm.to_string())

PX_HA = 30 * 30 / 10000
for i, n in enumerate(NAMES):
    net = (int((lc23 == i).sum()) - int((lc15 == i).sum())) * PX_HA
    if abs(net) > 1:
        print(f"  {n:10s} net change {net:+8.1f} ha")

## Step 6. Surface urban heat island and the equity of heat exposure

Splitting the urban area into formal and informal and comparing those is where this becomes policy evidence, because it produces a finding existing statistics cannot.

In [ ]:
for yr, lc, sc in [(2015, lc15, s15), (2023, lc23, s23)]:
    formal = float(np.nanmean(sc["lst_c"][lc == 3]))
    informal = float(np.nanmean(sc["lst_c"][lc == 4]))
    print(f"{yr}  formal {formal:.2f}C   informal {informal:.2f}C   gap {informal - formal:+.2f}C")

## Step 7. The NDVI to LST relationship

Quantifying this turns "planting trees cools the city" into an estimate a finance ministry can evaluate.

In [ ]:
nd = rs.ndvi(s23).ravel()
lst = s23["lst_c"].ravel()
print("NDVI to LST correlation:", round(float(np.corrcoef(nd, lst)[0, 1]), 3))

## Step 8. Aggregate to zones and hand over to Chapter 4

The moment a raster becomes a zonal average, Moran's I, LISA, and Gi* apply directly.

In [ ]:
h, w = s23["lst_c"].shape
zones = (np.arange(h)[:, None] // 10) * 100 + (np.arange(w)[None, :] // 10)
out = rs.zonal_stats(s23["lst_c"], zones)
print(out.head().round(2).to_string(index=False))
print("zones:", len(out))

---

**What to do next.** Compare against Section 7.4. To run this on your own city, start with `gee/01_landsat_lst.js` and change only the area of interest.